In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print('pytorch version:',torch.__version__)

# This gives us a nice way of summarising our model layers and parameters
!pip install torchinfo

# Make sure we can run on GPU as well as CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

---

You can use the function below to load a couple of the simple datasets available directly from `pytorch` via `torchvision`. There are two options for the `dataset_name` argument:
1. `mnist`: a dataset of handwritten digits from 0 - 9. These images are (28,28,1) in shape
2. `cifar10`: these are small colour images with shape (32,32,3) from ten different classes (plane, car, bird, cat, deer, dog, frog, horse, ship, truck)

In [ ]:
# This function loads the dataset (via torchvision) that we want:
#   - MNIST or CIFAR10
def load_dataset(dataset_name="mnist"):

    # Makes sure the tensors we get are correctly normalised and arranged
    transform = transforms.ToTensor()

    # MNIST first
    if dataset_name.lower() == "mnist":
        train_dataset = datasets.MNIST(
            root="./data",
            train=True,
            download=True,
            transform=transform,
        )
        test_dataset = datasets.MNIST(
            root="./data",
            train=False,
            download=True,
            transform=transform,
        )

        n_classes = 10

    # CIFAR10 (download can be very slow for some reason)
    elif dataset_name.lower() == "cifar10":
       # !mkdir -p data
       # !wget -nc https://www.hep.phy.cam.ac.uk/~lwhitehead/cifar-10-python.tar.gz -P data
       # !tar -xzf data/cifar-10-python.tar.gz -C data
        train_dataset = datasets.CIFAR10(
            root="./data",
            train=True,
            download=True,
            transform=transform,
        )

        test_dataset = datasets.CIFAR10(
            root="./data",
            train=False,
            download=True,
            transform=transform,
        )

        n_classes = 10

    else:
        raise ValueError(
            "Dataset must be one of: mnist or cifar10"
        )

    # Note that these shapes don't include the transform that moves
    # the channel dimension to be the second one
    # i.e (B, H, W, C) -> (B, C, H, W)
    # It is applied when accessing and member of the dataset, however
    print("Shape of training dataset =", train_dataset.data.shape)
    print("Shape of testing dataset  =", test_dataset.data.shape)

    # Plot a few examples
    fig, ax = plt.subplots(1, 5, figsize=(10, 2))

    for i in range(5):

        image, label = train_dataset[i]

        if dataset_name.lower() == "mnist":
            ax[i].imshow(image.squeeze(), cmap="gray")
        else:
            ax[i].imshow(image.permute(1, 2, 0))

        ax[i].axis("off")

    plt.show()

    return train_dataset, test_dataset, n_classes

---

Here we use the `load_data` function to load our dataset. In the
first instance we will use `mnist` since it is the simplest dataset and we can use a very simple CNN.

In [ ]:
# Load the input data.
# train_dataset contains the training images and labels
# test_dataset contains the testing images and labels
# Num_classes is the number of true classes
train_dataset, test_dataset, num_classes = load_dataset('mnist')

Before starting our CNN, let's make a simple MLP to see how well it does. MLPs consist of a number of fully connected layers. We need to make sure that we flatten the input in this case since we have images. We'll make a network with three dense layers (256, 128 and 64 neurons) interspersed with dropout layers (fraction 0.25), and then the final dense layer for classification.

* Flatten layer: `torch.nn.Flatten()`

* Fully-connected layer: `torch.nn.Linear(in_features, out_features)` where the in_features is the input size and out_features is how many neurons are in the layer. The final layer of the model needs have to have `out_features = num_classes`,

* Remember activations! `torch.nn.ReLU`. For the final layer we don't explicitly apply the `softmax` activation is it is included in the loss function in `pytorch`.

* Dropout layer: `torch.nn.Dropout(fraction)`

Printing a summary of the network should give you the following:
```
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
Sequential                               [1, 10]                   --
├─Flatten: 1-1                           [1, 784]                  --
├─Linear: 1-2                            [1, 256]                  200,960
├─ReLU: 1-3                              [1, 256]                  --
├─Dropout: 1-4                           [1, 256]                  --
├─Linear: 1-5                            [1, 128]                  32,896
├─ReLU: 1-6                              [1, 128]                  --
├─Dropout: 1-7                           [1, 128]                  --
├─Linear: 1-8                            [1, 64]                   8,256
├─ReLU: 1-9                              [1, 64]                   --
├─Dropout: 1-10                          [1, 64]                   --
├─Linear: 1-11                           [1, 10]                   650
==========================================================================================
Total params: 242,762
Trainable params: 242,762
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.24
==========================================================================================
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.97
Estimated Total Size (MB): 0.98
==========================================================================================
```

In [ ]:
import torch.nn as nn
import torchinfo

# Define our MLP: replace "None" with the corresponding layers as described
dummy_image, _ = train_dataset[0]
in_shape = (1, *dummy_image.shape)
in_c = in_shape[1]
in_h = in_shape[2]
in_w = in_shape[3]
print(in_shape)
mlp_model = nn.Sequential(
    # Flatten the 2D input into 1D for the fully connected layers
    None,
    # Fully connected layer with 256 neurons (need to calculate in_channels)
    None,
    # Relu activation function
    None,
    # Dropout with 25% of neurons disabled
    None,
    # Fully connected layer with 128 neurons
    None,
    # ReLU activation function
    None,
    # Dropout with 25% of neurons disabled
    None,
    # Fully connected layer with 128 neurons
    None,
    # ReLU activation function
    None,
    # Seventh layer: dropout with 25% of neurons disabled
    None,
    # Fully connected layer for classification into the number of classes
    # Note that we don't have the expected softmax activation here, this is
    # because pytorch (for some unknown reason) applies the softmax in the loss
    # function that we are going to use (CategoricalCrossEntropy)
    None
)

torchinfo.summary(mlp_model, input_size=in_shape)


Now we need to define the loss function and optimiser that we will use to perform the gradient descent optimisation.

* `torch.nn.CrossEntropyLoss` is the loss function for multi-category classification tasks
* `torch.optim.Adam(model.parameters(), lr=learning_rate)` is a choice of optimiser that can be used here. We need to give the parameters of our model and the learning rate as arguments.



In [ ]:
from torch.utils.data import DataLoader

# The batch size controls the number of images that are processed simultaneously
batch_size = 128

# Wrap out datasets in dataloaders now we know the batch size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# The learning rate (step size in gradient descent)
learning_rate = 0.001

# Categorical crossentropy loss function (which contains the softmax activation)
mlp_loss_function = None
# Adam optimiser using the learning rate defined above
mlp_optimiser = None

Now we are ready to train our MLP and run it on the MNIST dataset. We will write a function to do this. Don't worry too much about the details, but you can see the steps in so far as doing the forward pass, backward pass and updating the network weights via the optimiser.

In [ ]:
# Lets write a function to do the training loop as we'll want to make use of
# this in a couple of places
def train_model(network_model, n_epochs, train_loader, test_loader,
                optimiser, loss_function, device):
    network_model.to(device)
    for epoch in range(0, n_epochs):
        network_model.train()
        n_train_correct = 0
        n_train_total = 0
        for images, labels in train_loader:
            images.to(device)
            labels.to(device)
            optimiser.zero_grad()
            outputs = network_model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimiser.step()

            predictions = torch.argmax(outputs, dim=1)
            n_train_correct += (predictions == labels).sum().item()
            n_train_total += labels.size(0)

        network_model.eval()
        n_test_correct = 0
        n_test_total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images.to(device)
                labels.to(device)
                outputs = network_model(images)

                predictions = torch.argmax(outputs, dim=1)
                n_test_correct += (predictions == labels).sum().item()
                n_test_total += labels.size(0)

        print('Epoch', epoch,
              'train accuracy = ', n_train_correct / n_train_total,
              'test accuracy = ', n_test_correct / n_test_total)


In [ ]:
# The number of epochs that we want to train the network for
epochs = 5

train_model(mlp_model, epochs, train_loader, test_loader,
            mlp_optimiser, mlp_loss_function, device)

Let's define a couple of functions to look at some images that we classified incorrectly

In [ ]:
# Make a list of incorrect classifications
def find_incorrect_classifications(network_model, test_loader, device):

    incorrect_indices = []
    network_model.eval()

    image_index = 0
    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = network_model(images)

            # Select the class with the highest score
            predictions = torch.argmax(outputs, dim=1)

            for i in range(len(labels)):

                if predictions[i] != labels[i]:
                    incorrect_indices.append(
                        [
                            image_index + i,
                            predictions[i].item(),
                            labels[i].item()
                        ]
                    )

            image_index += len(labels)

    print(
        "Number of images that were incorrectly classified =",
        len(incorrect_indices)
    )

    return incorrect_indices

def draw_failure(dataset, incorrect_indices, index_to_show=0):

    image_index = incorrect_indices[index_to_show][0]

    image, label = dataset[image_index]

    prediction = incorrect_indices[index_to_show][1]

    print(
        "Incorrect classification for image",
        image_index,
        ": predicted =",
        prediction,
        "with true =",
        label
    )

    fig, ax = plt.subplots(1, 1)

    if image.shape[0] == 1:
        ax.imshow(image.squeeze(), cmap="gray")
    else:
        ax.imshow(image.permute(1, 2, 0))

    ax.axis("off")

And now lets look at the images

In [ ]:
mlp_failures = find_incorrect_classifications(mlp_model, test_loader, device)

In [ ]:
draw_failure(test_dataset, mlp_failures, 0)

---

Now we want to define a CNN. The basic building blocks you will need are:


*   Convolutional layers: `torch.nn.Conv2d(in_channels, out_channels, kernel_size, padding='same')`. Typical values of `kernel_size` are 3, 5, or 7. Various other arguments exist, but we just need to make use of these four.
*   Pooling layers: `torch.nn.MaxPool2d(kernel_size, stridge)` will perform a factor of 2 downsampling in the two dimensions of image
*   Dropout: keras.layers.Dropout(fraction) where fraction is the fraction of weights that are ignored. Typical values can be 0.25 or 0.5
*   Fully-connected layers: `torch.nn.Linear(in_features, out_features)` where `in_features` is the number of inputs and `out_features` is the number of nodes in the layer. The final layer of the CNN needs have to have `out_features = num_classes`
*   Flatten layer: This just converts and n-dimensional tensor into a vector. In this case we use it to present a fully connected output layer with a vector input


For the first CNN we are building, you will hopefully see the following output from the summary() command:

```
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
Sequential                               [1, 10]                   --
├─Conv2d: 1-1                            [1, 32, 28, 28]           320
├─ReLU: 1-2                              [1, 32, 28, 28]           --
├─MaxPool2d: 1-3                         [1, 32, 14, 14]           --
├─Dropout: 1-4                           [1, 32, 14, 14]           --
├─Flatten: 1-5                           [1, 6272]                 --
├─Linear: 1-6                            [1, 10]                   62,730
==========================================================================================
Total params: 63,050
Trainable params: 63,050
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.31
==========================================================================================
Input size (MB): 0.00
Forward/backward pass size (MB): 0.20
Params size (MB): 0.25
Estimated Total Size (MB): 0.46
==========================================================================================
```

In [ ]:
# Define our simple CNN model: replace "None" with the corresponding layers as described
dummy_image, _ = train_dataset[0]
in_shape = (1, *dummy_image.shape)
in_c = in_shape[1]
in_h = in_shape[2]
in_w = in_shape[3]
print(in_shape)
cnn_model = nn.Sequential(
    # A single convolutional layer with 32 output channels,
    # a kernel size of 3 and 'same' padding
    None,
    # ReLU activation function
    None,
    # Max pooling with kernel size 2 and stride 2
    None,
    # Dropout with 0.25 fraction
    None,
    # Flatten down into one dimension
    None,
    # Final linear layer - need to calculate the input size here!
    None
)

torchinfo.summary(cnn_model, in_shape)

In [ ]:
# Set up the model to train with the same hyperparameters as the MLP
# Categorical crossentropy loss function (which contains the softmax activation)
cnn_loss_function = None
# Adam optimiser using the learning rate defined above
cnn_optimiser = None

Now we can run our network on whichever data sample we requested. Initially on `mnist` we'll hopefully see that we can reach a very high accuracy.

In [ ]:
# Train the model using the training images and targets, and use the test
# images as the validation sample.
train_model(cnn_model, epochs, train_loader, test_loader,
            cnn_optimiser, cnn_loss_function, device)


Now let's look at some failures again

In [ ]:
cnn_failures = find_incorrect_classifications(cnn_model, test_loader, device)

In [ ]:
draw_failure(test_dataset, cnn_failures, 0)